### Data Ingetion ###

In [2]:
### Document Structure

from langchain_core.documents import Document

In [4]:
doc = Document(

    page_content="this is a page content to create RAG", 
    metadata= {
        "source": "example.txt",
        "author": "Bade Sahab",
        "pages": 1,
        "date_creation": "2026-03-22"

    }
)
doc

Document(metadata={'source': 'example.txt', 'author': 'Bade Sahab', 'pages': 1, 'date_creation': '2026-03-22'}, page_content='this is a page content to create RAG')

In [6]:
### Create a txt file
import os
os.makedirs("../data/text_files",exist_ok=True)


In [12]:
sample_texts={
    "../data/text_files/python_intro.txt":""" 

The Server-Sent Events (SSE) API enables pushing messages/updates from a server to the web page via HTTP connection.

Server-Sent Events - One Way Messaging
A server-sent event is when a web page automatically gets messages/updates from a server.

Normally, a web page has to request data from the server, but with server-sent events, the updates are pushed automatically.

Examples: Facebook/Twitter updates, stock market updates, news feeds, sport results, etc.

Browser Support
The numbers in the table specify the first browser version that fully support the Server-Sent Events API.

API					
SSE	6.0	79.0	6.0	5.0	11.5
Receive Server-Sent Event Notifications
The EventSource object is used to receive server-sent event notifications:

Example
<script>
const x = document.getElementById("result");
// Check browser support for SSE
if(typeof(EventSource) !== "undefined") {
  var source = new EventSource("demo_sse.php");
  source.onmessage = function(event) {
    x.innerHTML += event.data + "<br>";
  };
} else {
  x.innerHTML = "Sorry, no support for server-sent events.";
}
</script>
Example explained:

Create a new EventSource object, and specify the URL of the page sending the updates (in this example "demo_sse.php")
Each time an update is received, the onmessage event occurs
When an onmessage event occurs, put the received data into the element with id="result"


"""
}

for filepath,content in sample_texts.items():
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)
print("Sample text file created")

Sample text file created


In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding = "utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content=' \n\nThe Server-Sent Events (SSE) API enables pushing messages/updates from a server to the web page via HTTP connection.\n\nServer-Sent Events - One Way Messaging\nA server-sent event is when a web page automatically gets messages/updates from a server.\n\nNormally, a web page has to request data from the server, but with server-sent events, the updates are pushed automatically.\n\nExamples: Facebook/Twitter updates, stock market updates, news feeds, sport results, etc.\n\nBrowser Support\nThe numbers in the table specify the first browser version that fully support the Server-Sent Events API.\n\nAPI\t\t\t\t\t\nSSE\t6.0\t79.0\t6.0\t5.0\t11.5\nReceive Server-Sent Event Notifications\nThe EventSource object is used to receive server-sent event notifications:\n\nExample\n<script>\nconst x = document.getElementById("result");\n// Check browser support for SSE\nif(typeof(EventSource) !== "undefined") {\n  v

In [6]:
### Directory loader
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls= PyMuPDFLoader,
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'iText® 5.5.13.2 ©2000-2020 iText Group NV (AGPL-version)', 'creator': '', 'creationdate': '2026-03-06T08:29:44+05:30', 'source': '..\\data\\pdf\\sample_pdf.pdf', 'file_path': '..\\data\\pdf\\sample_pdf.pdf', 'total_pages': 57, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-03-06T08:29:44+05:30', 'trapped': '', 'modDate': "D:20260306082944+05'30'", 'creationDate': "D:20260306082944+05'30'", 'page': 0}, page_content='Statement for A/c XXXXXXXX2779 for the period 07-Mar-2025 to 06-Mar-2026\nCustomer Id\nXXXXXX18\nName\nPRANAV SATISH YELIKA\nPhone\n+919028950859\nAddress\nCO SATISH YELIKAR 1  1 28 P\nRH NO 02 GUTN 30 SHREE RESIDENCY\nAURANGABAD MAHARASHTRA IN 431008\nBranch Code\n2576\nBranch Name\nCHHATRAPATI\nSAMBHAJINAGAR JALNA ROAD\nProduct Code\n101\nProduct Name\nCANARA SB GENERAL\nIFSC Code\nCNRB0002576\nAddress\nPLOT NO.3, SURANA NAGAR,\nDate\nParticulars\nDeposits\nWithdrawals\nBalance\nOpening Balan

In [8]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer 
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Dell\Desktop\RAG\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
              model_name: HuggingFace model name for sentense embedding 
        """ 

        self.model_name = model_name
        self.model = None
        self._load_model()

    
    def _load_model(self):
        "Load sentenceTransformer model"
        try:
            print(f'Loading embedding model: {self.model_name}')
            self.model = SentenceTransformer(self.model_name)
            print(f'Model loaded successfully. Embedding dimention: {self.model.get_sentence_embedding_dimension()}')
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedding for a list of text

        Args:
        numpy array of embedding with shape (len(texts), embedding_dim)

        """

        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generate embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f'Generated embeddings with shape: {embeddings.shape}')
        return embeddings
    
    def get_embedding_dimention(self) -> int:
        """Get the embedding dimention of the model"""

        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    

## Initialize the embedding manager

Embedding_Manager = EmbeddingManager()

Embedding_Manager

              
              



Loading embedding model: all-MiniLM-L6-v2


c:\Users\Dell\Desktop\RAG\rag\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1764.03it/s]
BertModel LOAD REPORT from: sent

Model loaded successfully. Embedding dimention: 384


### Vector Store

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore